In [ ]:
import numpy as np
import tensorflow as tf

from os import listdir
from os.path import join
import re


def is_data_file(filename):
    return any(filename.endswith(extension) for extension in [".npy"])


def load_data(filepath):
    y = np.load(filepath)
    y = y.astype(np.float32)
    return y


def load_data2(filepath):
    y = np.load(filepath)
    y = y.astype(np.float32)
    return y


class DatasetFromFolder(tf.keras.utils.Sequence):
    def __init__(self, data_dir_input, data_dir_target, batch_size=256, shuffle=True, transform=None):
        super().__init__()
        self.data_filenames_input = [join(data_dir_input, x) for x in listdir(data_dir_input) if is_data_file(x)]
        self.data_filenames_target1 = [join(data_dir_target, x) for x in listdir(data_dir_target) if is_data_file(x)]
        self.data_filenames_input = sorted(self.data_filenames_input)
        self.data_filenames_target1 = sorted(self.data_filenames_target1)
        temp_target1 = self.data_filenames_target1
        temp_target = []
        for str1 in self.data_filenames_input:
            a = str1.split('/')[-1]
            a = re.split('[_ , .]', a)
            for str2 in temp_target1:
                b = str2.split('/')[-1]
                b = re.split('[_ , .]', b)
                if a[1] == b[2] and a[2] == b[3] and a[3] == b[4]:
                    temp_target.append(str2)
        self.data_filenames_target1 = temp_target
        self.transform = transform
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.data_filenames_input))
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __len__(self):
        return int(np.ceil(len(self.data_filenames_input) / self.batch_size))

    def __getitem__(self, index):
        start = index * self.batch_size
        end = (index + 1) * self.batch_size
        batch_indices = self.indices[start:end]
        inputs = []
        targets = []
        for idx in batch_indices:
            input_arr = load_data(self.data_filenames_input[idx])
            input_arr = input_arr[5::,]
            input_arr = np.transpose(input_arr, (1, 2, 0))
            target_arr = load_data2(self.data_filenames_target1[idx])
            target_arr = target_arr[1,]
            target_arr = target_arr[..., np.newaxis]
            inputs.append(input_arr)
            targets.append(target_arr)
        return np.stack(inputs, axis=0), np.stack(targets, axis=0)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)
